In [ ]:
import numpy as np
import PIL
import random
from sklearn.model_selection import cross_val_score

from PIL import Image
import matplotlib.pyplot as plt
from tensorflow.keras.regularizers import l2
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Reshape, Input, Conv2D, MaxPooling2D, Flatten, Dense, GRU
from tensorflow.keras.models import Model
import seaborn as sns
import glob
import sklearn.metrics as metrics
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, classification_report, f1_score, confusion_matrix, roc_curve, roc_auc_score,auc
from tensorflow.keras import layers, models
from tensorflow.keras.layers import Input, SeparableConv2D, BatchNormalization, Activation, Dense, Flatten, MaxPooling2D
from tensorflow.keras.layers import Add
from keras.layers import BatchNormalization
from keras.layers import ELU
from tensorflow.keras.layers import ReLU
import sys
from google.colab import files
import pandas as pd
from sklearn.preprocessing import label_binarize
import time
import numpy as np
import PIL
from PIL import Image
import matplotlib.pyplot as plt
from tensorflow.keras.regularizers import l2
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Reshape, Input, Conv2D, MaxPooling2D, Flatten,  GRU
from tensorflow.keras.models import Model
import seaborn as sns
import glob
import sklearn.metrics as metrics
from tensorflow.keras import layers, models
from tensorflow.keras.layers import Add
from keras.layers import BatchNormalization
from keras.layers import ELU
from tensorflow.keras.layers import ReLU
import sys
from google.colab import files
import pandas as pd
from sklearn.preprocessing import label_binarize
import time
from tensorflow.keras.applications import InceptionV3, ResNet50, EfficientNetB0, DenseNet201, InceptionResNetV2
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import (
    VGG16, ResNet50, MobileNetV2, EfficientNetB0, EfficientNetB3,
    DenseNet121, Xception, NASNetMobile, InceptionV3, ConvNeXtTiny
)
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import numpy as np
from tensorflow.keras.optimizers.schedules import ExponentialDecay
from itertools import combinations
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Model
from sklearn.decomposition import PCA
# from deap import base, creator, tools, algorithms


In [ ]:
#upload the dataset
x_train = np.load('E://datasets//HAM10000//ARTIFACT_FREE//SPLIT//X_train.npy')
x_test = np.load('E://datasets//HAM10000//ARTIFACT_FREE//SPLIT//X_test.npy')
y_train = np.load('E://datasets//HAM10000//ARTIFACT_FREE//SPLIT//y_train.npy')
y_test = np.load('E://datasets//HAM10000//ARTIFACT_FREE//SPLIT//y_test.npy')

In [ ]:
num_classes = y_train.shape[1]

In [ ]:
#PHMBCNN-GRU

In [ ]:
def residual_block(x, filters, y=None):
    if y is None:
        y = x  # Set y to x if y is not provided
    shortcut = y
    x = Conv2D(filters, (3, 3), padding='same', use_bias=False)(x)
    x = BatchNormalization()(x)
    x = ELU()(x)
    x = Conv2D(filters, (3, 3), padding='same', use_bias=False)(x)
    x = BatchNormalization()(x)
    return Add()([x, shortcut])

# Xception Block
def xception_block(x, filters):
    for _ in range(3):
        x = layers.SeparableConv2D(filters, (3, 3), padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.ELU()(x)
    return x

# Squeeze-Excite Block
def squeeze_excite_block(x, ratio=16):
    filters = x.shape[-1]
    se = layers.GlobalAveragePooling2D()(x)
    se = layers.Dense(filters // ratio, activation='elu')(se)
    se = layers.Dense(filters, activation='sigmoid')(se)
    return layers.multiply([x, se])

# MBConv Block (MobileNetV2)
def mbconv_block(x, filters, expansion_factor=6):
    input_filters = x.shape[-1]
    expanded_filters = input_filters * expansion_factor
    x = layers.Conv2D(expanded_filters, (1, 1), padding='same', activation='elu')(x)
    x = layers.DepthwiseConv2D((3, 3), padding='same')(x)
    x = layers.Conv2D(filters, (1, 1), padding='same', activation='linear')(x)
    return x

# Ghost Block
def ghost_block(x, filters):
    primary = layers.Conv2D(filters // 2, (1, 1), padding='same')(x)
    cheap_operation = layers.DepthwiseConv2D((3, 3), padding='same')(primary)
    return layers.concatenate([primary, cheap_operation])

# Fire Block (SqueezeNet)
def fire_block(x, squeeze_filters, expand_filters):
    x = layers.Conv2D(squeeze_filters, (1, 1), activation='elu')(x)
    expand1x1 = layers.Conv2D(expand_filters, (1, 1), activation='elu')(x)
    expand3x3 = layers.Conv2D(expand_filters, (3, 3), padding='same', activation='elu')(x)
    return layers.concatenate([expand1x1, expand3x3])

# Depthwise Separable Convolution Block
def dsc_block(x, filters):
    x = layers.SeparableConv2D(filters, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ELU()(x)
    return x

# Build the full model
def build_model(input_shape=(224, 224, 3), num_classes=7):
    inputs = layers.Input(shape=input_shape)

    # Initial Convolution and Pooling
    x = layers.Conv2D(32, (3, 3), padding='same', activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x32 = layers.MaxPooling2D(pool_size=(2, 2))(x)  # Shape: (64, 64, 32)

    x = layers.Conv2D(64, (3, 3), padding='same', activation='relu')(x32)
    x = layers.BatchNormalization()(x)
    x64 = layers.MaxPooling2D(pool_size=(2, 2))(x)  # Shape: (32, 32, 64)

    x = layers.Conv2D(128, (3, 3), padding='same', activation='relu')(x64)
    x = layers.BatchNormalization()(x)
    x128 = layers.MaxPooling2D(pool_size=(2, 2))(x)  # Shape: (16, 16, 128)

    x = layers.Conv2D(256, (3, 3), padding='same', activation='relu')(x128)
    x = layers.BatchNormalization()(x)
    x256 = layers.MaxPooling2D(pool_size=(2, 2))(x)  # Shape: (08, 08, 256)

    # Block 1: Residual + MBConv + Depthwise Separable Convolution
    x1 = residual_block(x32, 32)
    x1 = SeparableConv2D(32, (3, 3), padding='same',
                    depthwise_regularizer=l2(0.01),
                    pointwise_regularizer=l2(0.01))(x1)
    x1 = mbconv_block(x1, 32)
    mbconv1 = Conv2D(64,(3,3),padding="same")(x1)
    mbconv1 = layers.MaxPooling2D(pool_size=(2, 2))(mbconv1)
    x1 = SeparableConv2D(32, (3, 3), padding='same',
                    depthwise_regularizer=l2(0.01),
                    pointwise_regularizer=l2(0.01))(x1)
    x1 = dsc_block(x1,32)
    x1 = Conv2D(256, (3, 3), padding='same')(x1)
    x1 = BatchNormalization()(x1)
    x1 = layers.ReLU()(x1)

    # Block 2: Xception + res + dsc
    x2 = xception_block(x64, 64)
    x2 = SeparableConv2D(64, (3, 3), padding='same',
                    depthwise_regularizer=l2(0.01),
                    pointwise_regularizer=l2(0.01))(x2)
    x2 = residual_block(x2, 64,mbconv1)
    x2 = SeparableConv2D(64, (3, 3), padding='same',
                    depthwise_regularizer=l2(0.01),
                    pointwise_regularizer=l2(0.01))(x2)
    x2 = mbconv_block(x2, 64)
    x2 = Conv2D(256, (3, 3), padding='same')(x2)
    x2 = BatchNormalization()(x2)
    x2 = layers.ReLU()(x2)

    # Block 3: SE + MBConv + Depthwise Separable Convolution
    x3 = squeeze_excite_block(x128)
    x3 = SeparableConv2D(128, (3, 3), padding='same',
                    depthwise_regularizer=l2(0.01),
                    pointwise_regularizer=l2(0.01))(x3)
    x3 = mbconv_block(x3, 128)
    mbconv3 = Conv2D(256,(3,3),padding="same")(x3)
    mbconv3 = layers.MaxPooling2D(pool_size=(2, 2))(mbconv3)
    x3 = SeparableConv2D(128, (3, 3), padding='same',
                    depthwise_regularizer=l2(0.01),
                    pointwise_regularizer=l2(0.01))(x3)
    x3 = dsc_block(x3, 128)
    x3 = Conv2D(256, (3, 3), padding='same')(x3)
    x3 = BatchNormalization()(x3)
    x3 = layers.ReLU()(x3)

    # Block 4: MBConv + Residual + Depthwise Separable Convolution
    x4 = mbconv_block(x256, 256)
    x4 = SeparableConv2D(256, (3, 3), padding='same',
                    depthwise_regularizer=l2(0.01),
                    pointwise_regularizer=l2(0.01))(x4)
    x4 = residual_block(x4, 256,mbconv3)
    x4 = SeparableConv2D(256, (3, 3), padding='same',
                    depthwise_regularizer=l2(0.01),
                    pointwise_regularizer=l2(0.01))(x4)
    x4 = dsc_block(x4, 256)
    x4 = Conv2D(256, (3, 3), padding='same')(x4)
    x4 = BatchNormalization()(x4)
    x4 = layers.ReLU()(x4)

    # Concatenate all blocks
    x1=layers.MaxPooling2D(pool_size=(2, 2))(x1)
    x1=layers.MaxPooling2D(pool_size=(2, 2))(x1)
    x1=layers.MaxPooling2D(pool_size=(2, 2))(x1)
    x2=layers.MaxPooling2D(pool_size=(2, 2))(x2)
    x2=layers.MaxPooling2D(pool_size=(2, 2))(x2)
    x3=layers.MaxPooling2D(pool_size=(2, 2))(x3)
    concatenated = layers.concatenate([x1, x2, x3, x4], axis=-1)

    x = SeparableConv2D(1024, (3, 3), padding='same')(concatenated)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = SeparableConv2D(1024, (3, 3), padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)


    # Classifier
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512, activation='relu')(x)
    # x = layers.Dropout(0.5)(x)
    x = Reshape((-1,512))(x)
    gru = GRU(512)(x)
    x = layers.Dense(512, activation='relu')(gru)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs)
    
    optimizer = Adam(learning_rate=0.00001)
    model.compile(optimizer=optimizer, loss= 'categorical_crossentropy', metrics=['accuracy'])
    return model

# Build and compile the model
model = build_model(input_shape=(224, 224, 3), num_classes= num_classes)

# model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Print the model summary
model.summary()

total_layers = len(model.layers)
print("Total number of layers:", total_layers)

final_model = build_model(input_shape=(224, 224, 3), num_classes= num_classes)
history = final_model.fit(x_train,y_train, epochs=50, batch_size=32, validation_data=(x_test,y_test), verbose=0)